In [1]:
import os

In [2]:
%pwd

'c:\\Users\\lenovo\\Desktop\\Kidney-Disease-Prediction\\research'

In [11]:
os.chdir("..")

In [12]:
%pwd

'c:\\Users\\lenovo\\Desktop\\Kidney-Disease-Prediction'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL: str
    local_data_file:Path
    unzip_dir:Path

In [14]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [15]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        data_ingetion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir,
        )
        return data_ingetion_config

In [23]:
from cnnClassifier import logger
import gdown
import zipfile

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self) -> str:
        '''
        Fetch data from the url
        '''
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            # os.makedirs("artifacts/data_ingestion",exist_ok=True)
            os.makedirs(self.config.unzip_dir,exist_ok=True)
            logger.info(f"Downloading file from :[{dataset_url}] into :[{zip_download_dir}]")
            
            file_id = dataset_url.split('/')[-2]
            prefix = 'https://drive.google.com/uc?export=download&id='
            gdown.download(prefix+file_id, zip_download_dir)
            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
        except Exception as e:
            raise e
        
    def extract_zip_file(self) -> None:
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [24]:
# pipline
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e: 
    raise e
    

[2026-01-10 23:32:18,565: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-10 23:32:18,568: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-10 23:32:18,571: INFO: common: created directory at: artifacts]
[2026-01-10 23:32:18,574: INFO: 422915966: Downloading file from :[https://drive.google.com/file/d/1WqOq9gp-ZC2nY9QcoqTtfpCDn48Vr1o5/view?usp=sharing] into :[artifacts/data_ingestion/data.zip]]


Downloading...
From (original): https://drive.google.com/uc?export=download&id=1WqOq9gp-ZC2nY9QcoqTtfpCDn48Vr1o5
From (redirected): https://drive.google.com/uc?export=download&id=1WqOq9gp-ZC2nY9QcoqTtfpCDn48Vr1o5&confirm=t&uuid=bdcacbb6-085f-4296-a9e8-638de04234d3
To: c:\Users\lenovo\Desktop\Kidney-Disease-Prediction\artifacts\data_ingestion\data.zip
100%|██████████| 940M/940M [02:59<00:00, 5.25MB/s]   


[2026-01-10 23:35:23,615: INFO: 422915966: Downloaded data from https://drive.google.com/file/d/1WqOq9gp-ZC2nY9QcoqTtfpCDn48Vr1o5/view?usp=sharing into file artifacts/data_ingestion/data.zip]
